# Soft Voting vs Hard Voting in Ensemble Classifiers

This notebook demonstrates the difference between **soft voting** and **hard voting** in ensemble classification, specifically showing why RotationForest uses soft voting while proximity-based predictions use hard voting.

## Key Concepts

- **Hard Voting**: Each classifier casts one vote for its predicted class. The ensemble predicts the class with the most votes.
  - Formula: $\hat{y}_{hard} = \arg\max_c \sum_{m=1}^{M} \mathbb{1}[\hat{y}_m = c]$

- **Soft Voting**: Each classifier provides class probabilities. The ensemble averages these probabilities and predicts the class with the highest average probability.
  - Formula: $\hat{y}_{soft} = \arg\max_c \frac{1}{M}\sum_{m=1}^{M} p_m(c \mid x)$

These two methods can produce different predictions when probability distributions are skewed differently than vote counts.

## 1. Import Libraries and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Set random seed for reproducibility
np.random.seed(42)

## 2. Create a Multiclass Dataset

We'll generate a 3-class classification problem with moderate class overlap to demonstrate cases where hard and soft voting disagree.

In [ ]:
# Generate a 3-class dataset with moderate overlap
X, y = make_classification(
    n_samples=500,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_classes=3,
    n_clusters_per_class=2,
    class_sep=0.8,  # Moderate separation to create ambiguous cases
    random_state=42
)

# Split into train/test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Class distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")

## 3. Train Base Classifiers with Probability Support

We'll train three different classifiers that all support `predict_proba()` for soft voting.

In [ ]:
# Train three different classifiers
clf1 = LogisticRegression(max_iter=500, random_state=42)
clf2 = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=42)
clf3 = GaussianNB()

# Fit all models
clf1.fit(X_train, y_train)
clf2.fit(X_train, y_train)
clf3.fit(X_train, y_train)

# Get individual accuracies
print("Individual Model Accuracies:")
print(f"  Logistic Regression: {accuracy_score(y_test, clf1.predict(X_test)):.4f}")
print(f"  Random Forest:       {accuracy_score(y_test, clf2.predict(X_test)):.4f}")
print(f"  Gaussian Naive Bayes: {accuracy_score(y_test, clf3.predict(X_test)):.4f}")

## 4. Generate Per-Model Predictions and Probabilities

Let's collect the predictions and probability estimates from each model.

In [ ]:
# Get predictions from each model
preds_clf1 = clf1.predict(X_test)
preds_clf2 = clf2.predict(X_test)
preds_clf3 = clf3.predict(X_test)

# Get probability estimates from each model
proba_clf1 = clf1.predict_proba(X_test)
proba_clf2 = clf2.predict_proba(X_test)
proba_clf3 = clf3.predict_proba(X_test)

print(f"Predictions shape: {preds_clf1.shape}")
print(f"Probabilities shape: {proba_clf1.shape}")
print(f"\nExample predictions for first 5 test samples:")
print(f"  Clf1: {preds_clf1[:5]}")
print(f"  Clf2: {preds_clf2[:5]}")
print(f"  Clf3: {preds_clf3[:5]}")

## 5. Implement Hard Voting and Soft Voting from Scratch

### Hard Voting
Count how many models predict each class, then choose the class with the most votes:
$$\hat{y}_{hard} = \arg\max_c \sum_{m=1}^{M} \mathbb{1}[\hat{y}_m = c]$$

### Soft Voting
Average the predicted probabilities across all models, then choose the class with the highest average probability:
$$\hat{y}_{soft} = \arg\max_c \frac{1}{M}\sum_{m=1}^{M} p_m(c \mid x)$$

In [ ]:
# Stack predictions from all models
all_preds = np.column_stack([preds_clf1, preds_clf2, preds_clf3])

# HARD VOTING: Count votes for each class
def hard_vote(predictions):
    """
    predictions: (n_samples, n_models) array of predicted classes
    Returns: (n_samples,) array of majority vote predictions
    """
    n_samples = predictions.shape[0]
    n_classes = 3  # We know we have 3 classes
    vote_counts = np.zeros((n_samples, n_classes), dtype=int)
    
    for sample_idx in range(n_samples):
        for model_pred in predictions[sample_idx]:
            vote_counts[sample_idx, model_pred] += 1
    
    return np.argmax(vote_counts, axis=1), vote_counts

hard_preds, hard_vote_counts = hard_vote(all_preds)

# SOFT VOTING: Average probabilities across models
def soft_vote(probabilities_list):
    """
    probabilities_list: list of (n_samples, n_classes) probability arrays
    Returns: (n_samples,) array of soft vote predictions
    """
    # Average probabilities across all models
    avg_proba = np.mean(probabilities_list, axis=0)
    return np.argmax(avg_proba, axis=1), avg_proba

soft_preds, avg_proba = soft_vote([proba_clf1, proba_clf2, proba_clf3])

print(f"Hard voting accuracy: {accuracy_score(y_test, hard_preds):.4f}")
print(f"Soft voting accuracy: {accuracy_score(y_test, soft_preds):.4f}")

## 6. Compare with sklearn's VotingClassifier

Let's verify our implementations match sklearn's `VotingClassifier`.

In [ ]:
# Create voting classifiers
voting_hard = VotingClassifier(
    estimators=[('lr', clf1), ('rf', clf2), ('gnb', clf3)],
    voting='hard'
)

voting_soft = VotingClassifier(
    estimators=[('lr', clf1), ('rf', clf2), ('gnb', clf3)],
    voting='soft'
)

# Note: VotingClassifier refits the models, but since we use the same random states,
# predictions should match our custom implementations
voting_hard.fit(X_train, y_train)
voting_soft.fit(X_train, y_train)

sklearn_hard_preds = voting_hard.predict(X_test)
sklearn_soft_preds = voting_soft.predict(X_test)

# Verify our implementations match sklearn
print("Custom vs sklearn hard voting match:", np.array_equal(hard_preds, sklearn_hard_preds))
print("Custom vs sklearn soft voting match:", np.array_equal(soft_preds, sklearn_soft_preds))

print(f"\nsklearn Hard voting accuracy: {accuracy_score(y_test, sklearn_hard_preds):.4f}")
print(f"sklearn Soft voting accuracy: {accuracy_score(y_test, sklearn_soft_preds):.4f}")

## 7. Inspect Cases Where Hard and Soft Voting Disagree

This is the key demonstration: finding samples where hard and soft voting produce different predictions.

In [ ]:
# Find indices where hard and soft voting disagree
disagreements = np.where(hard_preds != soft_preds)[0]

print(f"Number of disagreements: {len(disagreements)} out of {len(y_test)} samples ({100*len(disagreements)/len(y_test):.1f}%)")

# Show details for first few disagreements
if len(disagreements) > 0:
    print("\n" + "="*80)
    print("DETAILED ANALYSIS OF DISAGREEMENTS")
    print("="*80)
    
    for i, idx in enumerate(disagreements[:5]):  # Show first 5 disagreements
        print(f"\n--- Sample {idx} (disagreement #{i+1}) ---")
        print(f"True label: {y_test[idx]}")
        print(f"Individual model predictions: {all_preds[idx]}")
        print(f"Hard vote counts: Class 0: {hard_vote_counts[idx, 0]}, Class 1: {hard_vote_counts[idx, 1]}, Class 2: {hard_vote_counts[idx, 2]}")
        print(f"Hard voting prediction: {hard_preds[idx]}")
        print(f"\nAverage probabilities: Class 0: {avg_proba[idx, 0]:.4f}, Class 1: {avg_proba[idx, 1]:.4f}, Class 2: {avg_proba[idx, 2]:.4f}")
        print(f"Soft voting prediction: {soft_preds[idx]}")
        print(f"\nPer-model probabilities:")
        print(f"  Clf1 (LogReg): {proba_clf1[idx]}")
        print(f"  Clf2 (RF):     {proba_clf2[idx]}")
        print(f"  Clf3 (GNB):    {proba_clf3[idx]}")
        print("\n" + "-"*80)
else:
    print("\nNo disagreements found between hard and soft voting on this dataset!")

## 8. Visualize Vote Counts vs Probability Sums

Let's create bar plots showing why hard and soft voting can disagree.

In [ ]:
if len(disagreements) > 0:
    # Visualize first 3 disagreement cases
    n_vis = min(3, len(disagreements))
    fig, axes = plt.subplots(n_vis, 2, figsize=(12, 4*n_vis))
    
    if n_vis == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_vis):
        idx = disagreements[i]
        
        # Left plot: Hard vote counts
        ax_left = axes[i, 0]
        classes = [0, 1, 2]
        vote_counts = hard_vote_counts[idx]
        bars_left = ax_left.bar(classes, vote_counts, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
        ax_left.set_xlabel('Class', fontsize=11)
        ax_left.set_ylabel('Vote Count', fontsize=11)
        ax_left.set_title(f'Sample {idx}: Hard Voting (predicts class {hard_preds[idx]})', fontsize=12, fontweight='bold')
        ax_left.set_xticks(classes)
        ax_left.set_ylim([0, 3.5])
        ax_left.grid(axis='y', alpha=0.3)
        
        # Highlight winning class
        bars_left[hard_preds[idx]].set_edgecolor('red')
        bars_left[hard_preds[idx]].set_linewidth(3)
        
        # Add value labels on bars
        for j, v in enumerate(vote_counts):
            ax_left.text(j, v + 0.05, str(v), ha='center', va='bottom', fontweight='bold')
        
        # Right plot: Average probabilities (summed across models)
        ax_right = axes[i, 1]
        avg_probs = avg_proba[idx]
        bars_right = ax_right.bar(classes, avg_probs, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
        ax_right.set_xlabel('Class', fontsize=11)
        ax_right.set_ylabel('Average Probability', fontsize=11)
        ax_right.set_title(f'Sample {idx}: Soft Voting (predicts class {soft_preds[idx]})', fontsize=12, fontweight='bold')
        ax_right.set_xticks(classes)
        ax_right.set_ylim([0, 1.0])
        ax_right.grid(axis='y', alpha=0.3)
        
        # Highlight winning class
        bars_right[soft_preds[idx]].set_edgecolor('red')
        bars_right[soft_preds[idx]].set_linewidth(3)
        
        # Add value labels on bars
        for j, v in enumerate(avg_probs):
            ax_right.text(j, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*80)
    print("KEY INSIGHT:")
    print("="*80)
    print("Hard voting counts each model's prediction equally (one vote per model).")
    print("Soft voting weights by confidence: a model's uncertain prediction (low probability)")
    print("contributes less than a confident one. This can flip the final prediction!")
else:
    print("No disagreements to visualize.")

## 9. Connection to RotationForest and Proximity-Based Predictions

### Why This Matters for RotationForest

**RotationForest** internally uses **soft voting**:
- Each tree in the ensemble produces a probability distribution over classes
- The final prediction is `argmax` of the **averaged probabilities** across all trees
- This is exactly what we demonstrated as "soft voting" above

**Proximity-based predictions** use **hard voting**:
- Proximities measure which training samples are "close" to a test sample
- Weighted vote: each training sample casts a vote for its class, weighted by proximity
- The final prediction is the class with the highest weighted vote count
- This is mathematically equivalent to "hard voting" (though weights are continuous, not binary)

### Example Scenario

Imagine a test sample where:
- Most trees place it in leaf nodes that contain **mostly class 0** training samples
- But those trees are **very uncertain** (probabilities like [0.51, 0.25, 0.24])
- A few trees place it in leaf nodes with **strong class 2** predictions (probabilities like [0.10, 0.05, 0.85])

**Soft voting (RotationForest)**: Might predict class 0 because the averaged probabilities favor it slightly

**Hard voting (Proximity)**: Might predict class 2 because more training samples in the leaf nodes belong to class 2

This is why we **cannot guarantee prediction equivalence** between RotationForest's `.predict()` method and proximity-based predictions. The two approaches aggregate information differently!

## Summary

This notebook demonstrated:

1. **Hard voting** = majority vote based on predicted class labels
2. **Soft voting** = averaged probabilities determine the final prediction
3. These two methods **can produce different predictions** when probability distributions are skewed
4. **RotationForest uses soft voting** (averages probabilities from trees)
5. **Proximity-based predictions use hard voting** (weighted class counts from training samples)
6. Therefore, **prediction equivalence is not guaranteed** between RotationForest's `.predict()` and proximity-based predictions

The key insight is that soft voting accounts for model confidence (probability values), while hard voting treats all predictions equally. This fundamental difference explains why the RotationForest classifier test in `test_sparse_proximity_mixin.py` does not enforce prediction equivalence.